# `klygo.config` (`inspection.py`): Collection views and structural comparison

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/config/Document_Config_Inspection.ipynb)


Inspect top-level or flattened keys, values, and `(key, value)` pairs, or compute leaf-level diffs between configurations or files.

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare workspace

In [1]:
from tempfile import TemporaryDirectory
from klygo import config, files

# Run this cell again to replace only this notebook's temporary workspace.
if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-config-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

Example directory: C:\Users\Mine\AppData\Local\Temp\klygo-config-tq0vuwy1


## Jump to an API

[`keys`](#keys) · [`values`](#values) · [`items`](#items) · [`diff`](#diff)

<a id="keys"></a>

### `config.keys`

Return top-level or flattened configuration keys.

**Signature:** `config.keys(config_data: dict[str, Any] | box.Box, flat: bool = False) -> list[str]`

In [2]:
sample = {"model": {"name": "yolov8n", "batch": 16}, "seed": 7}
print("Top-level keys:", config.keys(sample, flat=False))
print("Flattened keys:", config.keys(sample, flat=True))

Top-level keys: ['model', 'seed']
Flattened keys: ['model.name', 'model.batch', 'seed']


<a id="values"></a>

### `config.values`

Return top-level or flattened configuration values.

**Signature:** `config.values(config_data: dict[str, Any] | box.Box, flat: bool = False) -> list[Any]`

In [3]:
print("Top-level values:", config.values(sample, flat=False))
print("Flattened values:", config.values(sample, flat=True))

Top-level values: [{'name': 'yolov8n', 'batch': 16}, 7]
Flattened values: ['yolov8n', 16, 7]


<a id="items"></a>

### `config.items`

Return top-level or flattened `(key, value)` pairs.

**Signature:** `config.items(config_data: dict[str, Any] | box.Box, flat: bool = False) -> list[tuple[str, Any]]`

In [4]:
print("Flattened items:", config.items(sample, flat=True))
try:
    config.items(sample, flat="invalid")
except TypeError as exc:
    print("TypeError ->", exc)

Flattened items: [('model.name', 'yolov8n'), ('model.batch', 16), ('seed', 7)]
TypeError -> flat must be bool, got str


<a id="diff"></a>

### `config.diff`

Compare two configuration objects or files by flattened leaf path.

**Signature:** `config.diff(config1: dict[str, Any] | box.Box | str | pathlib.Path, config2: dict[str, Any] | box.Box | str | pathlib.Path) -> dict[str, dict[str, Any]]`

In [5]:
file_a = config.save(root / "v1.yaml", {"model": {"batch": 16, "lr": 0.01}}, overwrite=True, verbose=False)
file_b = config.save(root / "v2.yaml", {"model": {"batch": 32}, "seed": 42}, overwrite=True, verbose=False)
report = config.diff(file_a, file_b)
print("Added:", report["added"])
print("Removed:", report["removed"])
print("Modified:", report["modified"])

Added: {'seed': 42}
Removed: {'model.lr': 0.01}
Modified: {'model.batch': {'from': 16, 'to': 32}}


## Put it together

In [6]:
baseline = {"model": {"lr": 0.01, "layers": 18}}
candidate = {"model": {"lr": 0.005, "layers": 18, "dropout": 0.1}}
changes = config.diff(baseline, candidate)
print("Flat keys:", config.keys(candidate, flat=True))
print("Added:", changes["added"], "| Modified:", changes["modified"])

Flat keys: ['model.lr', 'model.layers', 'model.dropout']
Added: {'model.dropout': 0.1} | Modified: {'model.lr': {'from': 0.01, 'to': 0.005}}


## Remember

- Pass `flat=True` to `keys()`, `values()`, and `items()` to inspect leaf dot-paths across nested configuration trees.
- Use `diff(a, b)` to audit added, removed, and modified leaf keys before saving or deploying configuration changes.
- Next in `klygo.config`: continue with [Document_Config_Structure.ipynb](Document_Config_Structure.ipynb) or return to [Document_Config_Overview.ipynb](Document_Config_Overview.ipynb).
- Related Klygo modules: [`klygo.files`](../files/Document_Files_Overview.ipynb) · [`klygo.archive`](../archive/Document_Archive_Overview.ipynb).